# Pipeline Multi-Agent Event-Driven — Clip-Flow

Pipeline automatizado de conteudo **O Mago Mestre** com arquitetura multi-agente paralela.
LLM: **Google Gemini** (`gemini-2.5-flash`)

## Arquitetura (5 Layers)

```
L1 Monitoramento (paralelo) → L2 Broker (dedup) → L3 Curador (Gemini) → L4 Geracao (paralelo) → L5 Pos-Producao (paralelo)
```

| Layer | Componente | Funcao |
|-------|-----------|--------|
| L1 | MonitoringLayer | Google Trends + Reddit + RSS em paralelo via `asyncio.gather` |
| L2 | TrendBroker | `asyncio.Queue` + dedup via TrendAggregator |
| L3 | CuratorAgent | Gemini API seleciona melhores temas → WorkOrders |
| L4 | GenerationLayer | PhraseWorker (Gemini) + ImageWorker (backgrounds estaticos) |
| L5 | PostProductionLayer | CaptionWorker (Gemini) + HashtagWorker + QualityWorker |

## Como usar

1. Execute as celulas **na ordem** (Ctrl+F9 para rodar todas)
2. Celulas 1-5: **Setup** (instalar deps, configurar ambiente, verificar)
3. Celulas 6-7: **Testes rapidos** (frase + imagem avulsa)
4. Celulas 8-12: **Layer-by-layer** (debug — roda cada camada isoladamente)
5. Celulas 13-14: **Pipeline completo** (one-click — roda tudo de uma vez)

In [ ]:
# ============================================================
# Celula 1: Setup & Instalacao
# ============================================================
%pip install -q google-genai pillow trendspyg feedparser python-dotenv nest_asyncio requests beautifulsoup4

# Instalar fontes no Colab (Linux)
import os, platform

if platform.system() == "Linux":
    os.system("apt-get install -y fonts-dejavu-core > /dev/null 2>&1")
    print("✅ Fontes Linux instaladas")
else:
    print("ℹ️ Windows detectado — fontes do sistema serao usadas")

print("✅ Dependencias instaladas")

In [ ]:
# ============================================================
# Celula 2: Ambiente + Google Drive + API Key
# ============================================================
import os
import sys
import logging
from pathlib import Path

# ---------- Detectar ambiente ----------
try:
    from google.colab import userdata, drive
    IS_COLAB = True
    print("✅ Ambiente: Google Colab")
except ImportError:
    IS_COLAB = False
    print("ℹ️ Ambiente: Jupyter local")

# ---------- Paths do projeto ----------
if IS_COLAB:
    PROJECT_DIR = Path("/content/clip-flow")
    if not (PROJECT_DIR / "config.py").exists():
        !git clone -b estrutura-agents https://github.com/luigivivian/mago_generator.git {str(PROJECT_DIR)}
        print(f"✅ Repo clonado em: {PROJECT_DIR}")
    else:
        print(f"✅ Projeto ja existe em: {PROJECT_DIR}")
else:
    PROJECT_DIR = Path(".").resolve()
    if not (PROJECT_DIR / "config.py").exists():
        PROJECT_DIR = PROJECT_DIR.parent
    print(f"✅ Projeto local: {PROJECT_DIR}")

os.chdir(str(PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Confirmar config.py
assert (PROJECT_DIR / "config.py").exists(), f"❌ config.py NAO encontrado em: {PROJECT_DIR}"
print(f"✅ config.py encontrado")

# ---------- Google Drive (Colab only, opcional) ----------
if IS_COLAB:
    try:
        drive.mount("/content/drive")
        print("✅ Google Drive montado")
    except Exception:
        print("ℹ️ Google Drive nao montado (opcional)")

# ---------- API Key (Gemini) ----------
if IS_COLAB:
    try:
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
        print("✅ GOOGLE_API_KEY carregada dos Secrets")
    except Exception:
        if not os.environ.get("GOOGLE_API_KEY"):
            print("⚠️ GOOGLE_API_KEY nao encontrada!")
            print("   Configure em: Runtime > Secrets > GOOGLE_API_KEY")
            print("   Obtenha em: https://aistudio.google.com/apikey")
        else:
            print("✅ GOOGLE_API_KEY ja configurada via env")
else:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.environ.get("GOOGLE_API_KEY", "")
    print(f"🔑 GOOGLE_API_KEY: {'✅ configurada' if api_key else '❌ ausente — configure no .env'}")

# ---------- Diretorios ----------
os.makedirs(str(PROJECT_DIR / "output"), exist_ok=True)

# ---------- Logging global ----------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

# ---------- Event loop (Colab ja tem um rodando) ----------
import nest_asyncio
nest_asyncio.apply()
print("✅ nest_asyncio aplicado — pronto para await")

In [ ]:
# ============================================================
# Celula 3: Carregar config.py + Verificacao de Ambiente
# ============================================================
import importlib.util

# Importar config.py pelo path absoluto (evita conflito com modulo builtin 'config')
config_path = os.path.join(str(PROJECT_DIR), "config.py")
spec = importlib.util.spec_from_file_location("config", config_path)
config = importlib.util.module_from_spec(spec)
sys.modules["config"] = config
spec.loader.exec_module(config)

from config import ASSETS_DIR, BACKGROUNDS_DIR, FONTS_DIR, OUTPUT_DIR

print(f"📂 config.py carregado de: {config_path}")

# ---------- Verificacoes ----------
checks = []

# API Key
checks.append(("GOOGLE_API_KEY", "✅" if os.environ.get("GOOGLE_API_KEY") else "❌"))

# Diretorios
for name, path in [
    ("assets/", ASSETS_DIR),
    ("backgrounds/", BACKGROUNDS_DIR),
    ("fonts/", FONTS_DIR),
    ("output/", OUTPUT_DIR),
]:
    checks.append((name, "✅" if path.exists() else "❌"))

# Backgrounds
bgs = list(BACKGROUNDS_DIR.glob("**/*.jpg")) + list(BACKGROUNDS_DIR.glob("**/*.png")) + list(BACKGROUNDS_DIR.glob("**/*.webp"))
checks.append((f"Backgrounds: {len(bgs)}", "✅" if bgs else "⚠️ (placeholder sera criado)"))

# Fontes
fonts = list(FONTS_DIR.glob("*.ttf")) + list(FONTS_DIR.glob("*.otf"))
checks.append((f"Fontes customizadas: {len(fonts)}", "✅" if fonts else "⚠️ (fallback do sistema)"))

# Dependencias
for pkg_name, import_name in [
    ("google-genai", "google.genai"),
    ("Pillow", "PIL"),
    ("feedparser", "feedparser"),
    ("trendspyg", "trendspyg"),
]:
    try:
        __import__(import_name)
        checks.append((pkg_name, "✅"))
    except ImportError:
        checks.append((pkg_name, "❌"))

print("=" * 50)
print("VERIFICACAO DE AMBIENTE")
print("=" * 50)
for item, status in checks:
    print(f"  {status} {item}")
print("=" * 50)

# Preview backgrounds
if bgs:
    print(f"\n📂 Backgrounds ({len(bgs)} total):")
    for bg in sorted(bgs)[:10]:
        print(f"   • {bg.name}")
    if len(bgs) > 10:
        print(f"   ... e mais {len(bgs) - 10}")

In [ ]:
# ============================================================
# Celula 4: Patch Font Paths para Linux (Colab)
# ============================================================
# O image_maker.py tem paths Windows hardcoded para fontes.
# Monkey-patch para adicionar paths Linux sem alterar o codigo fonte.

import src.image_maker as image_maker_module
from PIL import ImageFont
from config import FONTS_DIR

def _load_font_universal(size: int) -> ImageFont.FreeTypeFont:
    """Versao patcheada de _load_font com fallback Linux + Windows."""
    # 1. Fontes customizadas em assets/fonts/ (prioridade)
    for ext in ("*.ttf", "*.otf"):
        for font_file in FONTS_DIR.glob(ext):
            try:
                return ImageFont.truetype(str(font_file), size)
            except Exception:
                continue

    # 2. Fallback por plataforma
    if IS_COLAB or platform.system() == "Linux":
        linux_fonts = [
            "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
            "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
            "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf",
            "/usr/share/fonts/truetype/ubuntu/Ubuntu-Bold.ttf",
        ]
        for font_path in linux_fonts:
            if os.path.exists(font_path):
                return ImageFont.truetype(font_path, size)
    else:
        windows_fonts = [
            "C:/Windows/Fonts/impact.ttf",
            "C:/Windows/Fonts/arialbd.ttf",
            "C:/Windows/Fonts/arial.ttf",
        ]
        for font_path in windows_fonts:
            if os.path.exists(font_path):
                return ImageFont.truetype(font_path, size)

    # 3. Fallback final
    return ImageFont.load_default()

# Aplicar monkey-patch
image_maker_module._load_font = _load_font_universal

# Teste
font = _load_font_universal(60)
nome = font.getname() if hasattr(font, "getname") else "default"
print(f"✅ Fonte carregada: {nome}")

In [ ]:
# ============================================================
# Celula 5: Teste Rapido — Frase Avulsa via Gemini
# ============================================================
from src.phrases import generate_phrases

TEMA_TESTE = "segunda-feira"  # ✏️ Altere aqui para testar outro tema

try:
    frases = generate_phrases(TEMA_TESTE, count=2)
    print(f"🧙 Frases sobre '{TEMA_TESTE}':")
    print("-" * 40)
    for i, frase in enumerate(frases, 1):
        print(f"  {i}. {frase}")
    print(f"\n✅ {len(frases)} frases geradas via Gemini API")
except Exception as e:
    frases = []
    print(f"❌ Erro na geracao de frases: {e}")
    print("   Verifique se GOOGLE_API_KEY esta configurada corretamente")

In [ ]:
# ============================================================
# Celula 6: Teste Rapido — Imagem Avulsa
# ============================================================
import random
from IPython.display import Image as IPyImage, display
from src.image_maker import create_image
from config import BACKGROUNDS_DIR

# Background aleatorio
bgs = list(BACKGROUNDS_DIR.glob("**/*.jpg")) + list(BACKGROUNDS_DIR.glob("**/*.png")) + list(BACKGROUNDS_DIR.glob("**/*.webp"))
bg_path = str(random.choice(bgs)) if bgs else None

# Frase: usa a gerada na celula anterior ou fallback
frase_teste = frases[0] if frases else "Cafe e o unico relacionamento estavel que eu mantenho"

try:
    img_path = create_image(frase_teste, bg_path)
    print(f"🖼️  Imagem gerada: {img_path}")
    display(IPyImage(filename=img_path, width=400))
except Exception as e:
    print(f"❌ Erro na geracao de imagem: {e}")

---
## Execucao Layer-by-Layer (Debug)

Cada celula abaixo roda uma camada do pipeline isoladamente.
Os dados fluem de uma celula para a proxima:

```
events → ranked_events → work_orders → packages → packages (enriquecidos)
```

> **Dica:** Use estas celulas para debugar cada camada individualmente.
> Para rodar o pipeline completo de uma vez, pule para a **Celula 12**.

In [ ]:
# ============================================================
# Celula 7: Layer 1 — Monitoramento (paralelo)
# ============================================================
import asyncio
from collections import Counter
from config import PIPELINE_GOOGLE_TRENDS_GEO, PIPELINE_REDDIT_SUBREDDITS, PIPELINE_RSS_FEEDS

from src.pipeline.agents.async_base import SyncAgentAdapter
from src.pipeline.agents.google_trends import GoogleTrendsAgent
from src.pipeline.agents.reddit_memes import RedditMemesAgent
from src.pipeline.agents.rss_feeds import RSSFeedAgent
from src.pipeline.monitoring import MonitoringLayer

# Instanciar agentes sync e wrapar em async
sync_agents = [
    GoogleTrendsAgent(geo=PIPELINE_GOOGLE_TRENDS_GEO),
    RedditMemesAgent(subreddits=PIPELINE_REDDIT_SUBREDDITS),
    RSSFeedAgent(feeds=PIPELINE_RSS_FEEDS),
]
async_agents = [SyncAgentAdapter(a) for a in sync_agents]

monitoring = MonitoringLayer(async_agents)
events = await monitoring.fetch_all()

# Resumo por fonte
source_counts = Counter(e.source.value for e in events)

print("=" * 50)
print(f"LAYER 1: MONITORAMENTO — {len(events)} trends coletados")
print("=" * 50)
for source, count in source_counts.most_common():
    print(f"  📡 {source}: {count} trends")
print()
print("Top 10 trends:")
for i, e in enumerate(sorted(events, key=lambda x: x.score, reverse=True)[:10], 1):
    print(f"  {i:2d}. [{e.source.value:15s}] (score={e.score:.2f}) {e.title[:60]}")

In [ ]:
# ============================================================
# Celula 8: Layer 2 — Broker (dedup + ranking)
# ============================================================
from src.pipeline.broker import TrendBroker

try:
    _has_events = bool(events)
except NameError:
    _has_events = False

if not _has_events:
    print("⚠️ Nenhum evento disponivel. Execute a celula 7 primeiro.")
else:
    broker = TrendBroker()

    # Ingerir eventos (dedup + boost multi-fonte)
    queued = await broker.ingest(events)

    # Drenar fila rankeada
    ranked_events = await broker.drain(max_items=20)

    print("=" * 50)
    print(f"LAYER 2: BROKER")
    print("=" * 50)
    print(f"  Eventos recebidos:     {len(events)}")
    print(f"  Eventos apos dedup:    {queued}")
    print(f"  Eventos drenados:      {len(ranked_events)}")
    print()
    print("Top trends apos dedup:")
    for i, e in enumerate(ranked_events[:10], 1):
        multi = f" [MULTI-FONTE x{e.sources_count}]" if e.sources_count > 1 else ""
        print(f"  {i:2d}. (score={e.score:.2f}) {e.title[:55]}{multi}")

In [ ]:
# ============================================================
# Celula 9: Layer 3 — Curador (Gemini API → WorkOrders)
# ============================================================
from src.pipeline.curator import CuratorAgent

curator = CuratorAgent()

# ✏️ Quantas imagens gerar (ajuste aqui)
IMAGES_PER_RUN = 3

topics_count = min(IMAGES_PER_RUN, len(ranked_events))

try:
    work_orders = await curator.curate(ranked_events, count=topics_count)

    print("=" * 50)
    print(f"LAYER 3: CURADOR — {len(work_orders)} WorkOrders emitidos")
    print("=" * 50)
    for wo in work_orders:
        print(f"\n  📋 Order: {wo.order_id}")
        print(f"     Topic:    {wo.gandalf_topic}")
        print(f"     Humor:    {wo.humor_angle}")
        print(f"     Cena:     {wo.situacao_key}")
        print(f"     Score:    {wo.relevance_score:.2f}")
        print(f"     Trend:    {wo.trend_event.title[:50]}")
except Exception as e:
    work_orders = []
    print(f"❌ Curador falhou: {e}")
    print("   Verifique se GOOGLE_API_KEY esta configurada e a API esta acessivel")

In [ ]:
# ============================================================
# Celula 10: Layer 4 — Geracao (frases + imagens em paralelo)
# ============================================================
from IPython.display import Image as IPyImage, display
from src.pipeline.workers.phrase_worker import PhraseWorker
from src.pipeline.workers.image_worker import ImageWorker
from src.pipeline.workers.generation_layer import GenerationLayer

generation = GenerationLayer(
    phrase_worker=PhraseWorker(),
    image_worker=ImageWorker(use_comfyui=False),  # Sem ComfyUI no Colab
    phrases_per_topic=1,
)

if not work_orders:
    print("⚠️ Nenhum work order disponivel. Execute a celula 9 primeiro.")
else:
    packages = await generation.process(work_orders)

    print("=" * 50)
    print(f"LAYER 4: GERACAO — {len(packages)} pacotes gerados")
    print("=" * 50)
    for i, pkg in enumerate(packages, 1):
        print(f"\n  🖼️  Pacote {i}:")
        print(f"     Frase:  {pkg.phrase[:60]}...")
        print(f"     Topic:  {pkg.topic}")
        print(f"     Imagem: {pkg.image_path}")

    # Preview das imagens
    for pkg in packages:
        if os.path.exists(pkg.image_path):
            print(f"\n--- {pkg.topic} ---")
            display(IPyImage(filename=pkg.image_path, width=350))

In [ ]:
# ============================================================
# Celula 11: Layer 5 — Pos-Producao (caption + hashtags + quality)
# ============================================================
from src.pipeline.workers.post_production import PostProductionLayer

post_production = PostProductionLayer()

try:
    _has_packages = bool(packages)
except NameError:
    _has_packages = False
    packages = []

if not _has_packages:
    print("⚠️ Nenhum pacote disponivel. Execute a celula 10 primeiro.")
else:
    packages = await post_production.enhance(packages)

    print("=" * 50)
    print(f"LAYER 5: POS-PRODUCAO — {len(packages)} pacotes enriquecidos")
    print("=" * 50)
    for i, pkg in enumerate(packages, 1):
        print(f"\n  📦 Pacote {i}: {pkg.topic}")
        print(f"     Frase:    {pkg.phrase[:60]}")
        print(f"     Quality:  {pkg.quality_score:.2f}")
        if pkg.caption:
            print(f"     Caption:  {pkg.caption[:80]}...")
        else:
            print(f"     Caption:  (vazio)")
        if pkg.hashtags:
            print(f"     Hashtags: {' '.join(pkg.hashtags[:5])}...")
        else:
            print(f"     Hashtags: (vazio)")

---
## Pipeline Completo — One-Click

Roda todas as 5 layers de uma vez usando o `AsyncPipelineOrchestrator`.

> **Nota:** Use esta celula quando quiser rodar o pipeline inteiro sem debugar layer por layer.

In [ ]:
# ============================================================
# Celula 12: Pipeline Completo — One-Click
# ============================================================
from src.pipeline.async_orchestrator import AsyncPipelineOrchestrator

# ============================================================
# ✏️ CONFIGURE AQUI
# ============================================================
IMAGES_PER_RUN = 3        # quantas imagens gerar
PHRASES_PER_TOPIC = 1     # frases por tema
USE_COMFYUI = False       # False no Colab (sem GPU local)
# ============================================================

orchestrator = AsyncPipelineOrchestrator(
    images_per_run=IMAGES_PER_RUN,
    phrases_per_topic=PHRASES_PER_TOPIC,
    use_comfyui=USE_COMFYUI,
)

result = await orchestrator.run()

# Dashboard de metricas
duration = (result.finished_at - result.started_at).total_seconds() if result.finished_at else 0

print("\n" + "=" * 60)
print("🧙 PIPELINE MULTI-AGENTE — RESULTADO FINAL")
print("=" * 60)
print(f"  ⏱️  Duracao:             {duration:.1f}s")
print(f"  📡 Trends coletados:    {result.trends_fetched}")
print(f"  📥 Eventos enfileirados: {result.trend_events_queued}")
print(f"  📋 Work orders:         {result.work_orders_emitted}")
print(f"  🖼️  Imagens geradas:     {result.images_generated}")
print(f"  📦 Pacotes produzidos:  {result.packages_produced}")
if result.errors:
    print(f"  ⚠️  Erros:              {len(result.errors)}")
    for err in result.errors:
        print(f"       - {err}")
print("=" * 60)

In [ ]:
# ============================================================
# Celula 13: Visualizacao Final + Export
# ============================================================
from IPython.display import Image as IPyImage, display
import shutil
import zipfile
from datetime import datetime

# Usar packages do pipeline completo (celula 12) ou do layer-by-layer (celula 11)
try:
    final_packages = result.content if result.content else packages
except NameError:
    try:
        final_packages = packages
    except NameError:
        final_packages = []
        print("⚠️ Nenhum pacote disponivel. Execute o pipeline primeiro (celula 12 ou celulas 7-11).")

if final_packages:
    print(f"🖼️  {len(final_packages)} imagens geradas\n")

    for i, pkg in enumerate(final_packages, 1):
        if os.path.exists(pkg.image_path):
            print(f"{'=' * 50}")
            print(f"📦 Pacote {i}: {pkg.topic}")
            print(f"   Frase: {pkg.phrase}")
            if pkg.caption:
                print(f"   Caption: {pkg.caption[:100]}...")
            if pkg.hashtags:
                print(f"   Hashtags: {' '.join(pkg.hashtags[:8])}")
            print(f"   Quality: {pkg.quality_score:.2f}")
            display(IPyImage(filename=pkg.image_path, width=400))
            print()

# ============================================================
# Export — escolha o modo
# ============================================================
# "drive" — copiar para Google Drive
# "zip"   — compactar e baixar (Colab only)
# "ambos" — Drive + zip
MODO_EXPORT = "drive"  # ✏️ Altere aqui

DRIVE_OUTPUT = "/content/drive/MyDrive/clip-flow/output/pipeline"

if final_packages:
    # Export para Drive
    if "drive" in MODO_EXPORT and os.path.exists("/content/drive/MyDrive"):
        os.makedirs(DRIVE_OUTPUT, exist_ok=True)
        copied = 0
        for pkg in final_packages:
            if os.path.exists(pkg.image_path):
                dest = os.path.join(DRIVE_OUTPUT, os.path.basename(pkg.image_path))
                shutil.copy2(pkg.image_path, dest)
                copied += 1
        print(f"\n✅ {copied} imagens copiadas para Google Drive: {DRIVE_OUTPUT}")

    # Export como ZIP
    if "zip" in MODO_EXPORT:
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        zip_path = Path(f"pipeline_output_{ts}.zip")
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for pkg in final_packages:
                if os.path.exists(pkg.image_path):
                    zf.write(pkg.image_path, os.path.basename(pkg.image_path))
        size_mb = zip_path.stat().st_size / 1024 / 1024
        print(f"\n✅ ZIP criado: {zip_path} ({size_mb:.1f} MB)")
        if IS_COLAB:
            from google.colab import files
            files.download(str(zip_path))
            print("📥 Download iniciado")

    if not IS_COLAB and "drive" in MODO_EXPORT:
        print("\nℹ️ Google Drive so disponivel no Colab — imagens salvas em output/")